# Baseline 04c — Ablation de bloques de features

Mide el aporte incremental de cada bloque del vector fused. Para cada conjunto de columnas entrenamos XGBoost (mas LightGBM como sanity) con identica spatial CV 5-fold y reportamos F1-macro + delta vs `full`.

Conjuntos canonicos evaluados:

- `full` — todas las features numericas disponibles.
- `no_geom` — `full` sin las 3 columnas `geom_*`.
- `no_geom_no_era5_srtm` — adicionalmente sin `era5_*` ni `srtm_*`.
- `alphaearth_only` — solo las 64 dimensiones `ae_*`.
- `phenology_only` — 8 fenologicas + 24 FFT NDVI.
- `geom_only` — solo `geom_*` (test cuantitativo de leakage espacial).

**Fix US-023-preview**: la deteccion de columnas AlphaEarth ahora tolera variantes (`ae_*`, `emb_*`, `dim_*`, `alphaearth_*`) y el conjunto `alphaearth_only` ya no aparece con `n_features=0` ni NaN cuando hay AE en el dataset.

In [ ]:
FEATURES_PATH = "data/test_fixtures/feature_selection_parcels_subset.parquet"
PARCELS_GEOPARQUET = "data/processed/pastis_parcels_full.geoparquet"
FIGURES_SUBDIR = "us-023-preview/04c_baseline"
REPORTS_SUBDIR = "baseline/04c_baseline"
K_FOLDS = 5
BUFFER_KM = 1.0
MAX_SAMPLES = None  # None = dataset completo; reducir para CI rapido.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Bootstrap: localizar el repo root buscando pyproject.toml
_HERE = Path.cwd().resolve()
for _candidate in (_HERE, *_HERE.parents):
    if (_candidate / "pyproject.toml").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from ml.utils.notebook_bootstrap import setup_notebook
from IPython.display import Markdown, display

env = setup_notebook(
    figures_subdir=FIGURES_SUBDIR,
    reports_subdir=REPORTS_SUBDIR,
)
display(Markdown(env.summary_markdown()))


## Carga del dataset + ablation

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from ml.utils.baseline_notebook_helpers import (
    load_features_dataset_with_meta,
    run_ablation_and_persist,
)
from ml.eval.reencuadre_plots import (
    plot_ablation_bars,
    plot_geom_leakage_comparison,
)

df = load_features_dataset_with_meta(
    path=FEATURES_PATH,
    parcels_geoparquet=PARCELS_GEOPARQUET,
)
display(Markdown(f'Dataset: `{df.height:,}` parcelas x `{df.width}` cols'))

ablation_table, parquet_path = run_ablation_and_persist(
    df,
    output_dir=env.reports_dir,
    models=('xgb',),
    k_folds=K_FOLDS,
    buffer_km=BUFFER_KM,
    max_samples=MAX_SAMPLES,
)
display(Markdown(f'**Tabla ablation**: `{parquet_path.relative_to(env.repo)}`'))
display(ablation_table)


## Graficas: barras de F1 por conjunto + comparativa geom

In [ ]:
from ml.eval.feature_ablation import FeatureAblationResult

results = [
    FeatureAblationResult(
        feature_set=row['feature_set'],
        model_kind=row['model'],
        f1_macro=row['f1_macro'] if row['f1_macro'] is not None else float('nan'),
        f1_weighted=row['f1_weighted'] if row['f1_weighted'] is not None else float('nan'),
        miou=row['miou'] if row['miou'] is not None else float('nan'),
        n_features=row['n_features'],
        delta_vs_full=row['delta_vs_full'] if row['delta_vs_full'] is not None else float('nan'),
    )
    for row in ablation_table.iter_rows(named=True)
]

fig_abl = plot_ablation_bars(results, title='F1-macro por conjunto de features (04c)')
fig_abl.savefig(env.figures_dir / 'ablation_bars.png', bbox_inches='tight')
display(fig_abl)
plt.close(fig_abl)

fig_geom = plot_geom_leakage_comparison(results)
fig_geom.savefig(env.figures_dir / 'geom_leakage.png', bbox_inches='tight')
display(fig_geom)
plt.close(fig_geom)


## Conclusiones

**Lectura honesta de la ablation**:

- El conjunto `full` define la referencia. El delta de `no_geom` vs `full` cuantifica el aporte (o leakage) de las columnas geometricas: si delta ~0 significa que `geom_*` no aporta señal agronomica; si delta es positivo, descartarlas mejora porque introducian ruido.

- El conjunto `geom_only` es el **test cuantitativo de leakage**: F1-macro < 0.10 confirma que area/perimetro/elongacion por si solas no permiten clasificar cultivos — el modelo no puede aprender clase a partir de geometria.

- `alphaearth_only` muestra cuanto del baseline viene de los 64 embeddings del Foundation Model. Si la diferencia entre `alphaearth_only` y `full` es pequeña, los demas bloques no estan agregando mucho mas alla del FM.

## Lo que sigue

- `05_reencuadre_fenologico.ipynb` amplia esta tabla con los bloques opcionales (FarSLIP, pheno_text Gemini real, firma espectral REP) materializados desde el propio notebook si no existen.
- `Avance3.Equipo17.ipynb` consume `ablation_table.parquet` para decidir el conjunto ganador.